In [2]:
import xarray as xr
import sys 
import os
os.chdir("../")
import lightning as L
from pytorch_lightning.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
from src.models.model import SimpleMLP
from src.data.datamodule import CustomDataModule
from src.models.lightning_module import RainfallRegressionModel
from src.utils.config import load_config
import argparse
import os

In [3]:
config = load_config('configs/experiments/default.yml')

wandb_logger = WandbLogger(
    project=config['logging']['project'],
    name=config['logging'].get('run_name'),
    config=config,
    save_dir=config['logging']['base_dir']
)

run_id = wandb_logger.experiment.id
out_dir = f"{config['logging']['base_dir']}/{config['logging']['project']}/{run_id}"

datamodule = CustomDataModule(
    data_in_path=config['data']['input_path'],
    data_target_path=config['data']['target_path'],
    batch_size=config['data']['batch_size']
)

image_size = datamodule.image_shape[1]*datamodule.image_shape[2]

model = SimpleMLP(input_size=image_size, 
                    hidden_size=config['model']['hidden_size'], 
                    target_size=1)

lightning_model = RainfallRegressionModel(model, learning_rate=config['trainer']['learning_rate'])

checkpoint_callback = ModelCheckpoint(
    dirpath=os.path.join(out_dir, "checkpoints"),
    filename="best",
    monitor="val_loss",
    save_top_k=1,
    mode="min",
)

trainer = L.Trainer(
    max_epochs=config['trainer']['max_epochs'],
    logger=wandb_logger,
    default_root_dir=out_dir,
    callbacks=[checkpoint_callback],
)

trainer.fit(lightning_model, datamodule=datamodule)
trainer.validate(lightning_model, datamodule=datamodule)




wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/robin/.netrc.
wandb: Currently logged in as: rguilcas (rguilcas-university-of-bergen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name      | Type      | Params | Mode  | FLOPs
--------------------------------------------------------
0 | model     | SimpleMLP | 262 K  | train | 0    
1 | criterion | MSELoss   | 0      | train | 0    
--------------------------------------------------------
262 K     Trainable params
0         Non-trainable params
262 K     Total params
1.049     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/anaconda3/envs/bccr-ml/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/opt/anaconda3/envs/bccr-ml/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.
/opt/anaconda3/envs/bccr-ml/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        val_loss            24.238544464111328
         val_r2              0.724158525466919
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'val_loss': 24.238544464111328, 'val_r2': 0.724158525466919}]

In [5]:
import torch
predictions = torch.cat(trainer.predict(lightning_model, datamodule=datamodule), dim=0)
predictions

/opt/anaconda3/envs/bccr-ml/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

tensor([[ 1.2836e+01],
        [ 2.3209e+00],
        [ 6.8828e+00],
        [ 2.2567e+01],
        [ 1.3726e+01],
        [ 1.2992e+01],
        [-1.0594e+00],
        [ 6.8680e+00],
        [ 9.3069e+00],
        [ 2.2206e+01],
        [-9.1041e-01],
        [-1.8197e+00],
        [ 9.7280e-01],
        [ 6.0106e-01],
        [ 7.3466e+00],
        [ 5.2407e+00],
        [ 1.9451e+00],
        [ 1.9866e+01],
        [ 1.3800e+01],
        [-1.3794e+00],
        [ 1.1262e+00],
        [ 3.2224e-01],
        [ 3.1311e+01],
        [ 1.0339e+01],
        [ 1.2682e+01],
        [ 9.9082e-01],
        [ 6.5546e-01],
        [ 1.8238e+00],
        [ 1.0184e+01],
        [-8.1883e-02],
        [ 3.8154e+00],
        [ 7.0489e+00],
        [-3.1737e+00],
        [ 1.1175e+01],
        [ 2.3510e+01],
        [ 6.0145e+00],
        [ 3.4199e+00],
        [ 7.7495e+00],
        [ 1.0626e+01],
        [ 1.0609e+01],
        [ 1.7013e+00],
        [ 1.0227e+00],
        [ 2.8333e+00],
        [ 3

In [16]:
from src.models.model import SimpleMLP
from src.data.dataset import CustomDataset
from src.data.datamodule import CustomDataModule
from src.models.lightning_module import RainfallRegressionModel
from src.utils.config import load_config

In [18]:
load_config('../configs/experiments/default.yml')

{'model': {'input_channels': 1, 'hidden_size': 128},
 'data': {'input_path': 'data/raw/msl_data_northatlantic.nc',
  'target_path': 'data/raw/pr_era5_daily_westnorway_1940-2024.nc',
  'batch_size': 128,
  'num_workers': 4},
 'trainer': {'max_epochs': 50,
  'learning_rate': 0.001,
  'accelerator': 'gpu',
  'devices': 1},
 'logging': {'project': 'extreme-rainfall', 'run_name': 'baseline'}}

In [3]:
datamodule = CustomDataModule(
    data_in_path="../data/raw/msl_data_northatlantic.nc",
    data_out_path="../data/raw/pr_era5_daily_westnorway_1940-2024.nc",
    batch_size=128)

In [4]:
dataset = CustomDataset(
    ds_in_path="../data/raw/msl_data_northatlantic.nc",
    ds_out_path="../data/raw/pr_era5_daily_westnorway_1940-2024.nc",)

In [ ]:
model = SimpleMLP(input_size=datamodule.image_shape[1]*datamodule.image_shape[2], hidden_size=64, output_size=1)
lightning_model = RainfallRegressionModel(model, learning_rate=1e-3)
trainer = L.Trainer(max_epochs=20)
lightning_model.train()
trainer.fit(lightning_model, datamodule=datamodule)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name      | Type      | Params | Mode  | FLOPs
--------------------------------------------------------
0 | model     | SimpleMLP | 131 K  | train | 0    
1 | criterion | MSELoss   | 0      | train | 0    
--------------------------------------------------------
131 K     Trainable params
0         Non-trainable params
131 K     Total params
0.524     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/anaconda3/envs/bccr-ml/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/opt/anaconda3/envs/bccr-ml/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/opt/anaconda3/envs/bccr-ml/lib/python3.13/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (24) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.


In [9]:
trainer.validate(lightning_model, datamodule=datamodule)

/opt/anaconda3/envs/bccr-ml/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        val_loss             31.75809669494629
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'val_loss': 31.75809669494629}]